<a href="https://colab.research.google.com/github/mimomaina/Career-Path-Recommendation-System/blob/main/(Updated)_Career_recommendation_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preprocessing

In [36]:
# !pip install beautifulsoup4 pandas numpy
from bs4 import BeautifulSoup
import pandas as pd
import re


In [37]:
def clean_text(text):
    """Cleans HTML tags and special characters from text"""
    if pd.isna(text):  # Handle NaN values
        return ""

    # Remove HTML tags
    text = BeautifulSoup(text, "html.parser").get_text()

    # Remove special symbols & multiple spaces
    text = re.sub(r"[\r\n\t]", " ", text)  # Remove line breaks & tabs
    text = re.sub(r"\s+", " ", text).strip()  # Remove extra spaces

    return text


In [38]:
# Load dataset
df = pd.read_csv("/content/tech_industry_dataset.csv")

# Select relevant columns
df = df[['Job Title', 'Job Description', 'Responsibilities', 'skills']]

# Apply text cleaning function
df['Job Description'] = df['Job Description'].apply(clean_text)
df['Responsibilities'] = df['Responsibilities'].apply(clean_text)
df['skills'] = df['skills'].apply(clean_text)

# Combine description + skills for embedding
df['combined_text'] = df['Job Description'] + " " + df['skills']


An embedding is a way of turning words or sentences into numbers (vectors) so that computers can understand and compare them.

Similar words/phrases have embeddings that are close together.
Unrelated words have embeddings that are far apart.

In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 292167 entries, 0 to 292166
Data columns (total 5 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   Job Title         292167 non-null  object
 1   Job Description   292167 non-null  object
 2   Responsibilities  292167 non-null  object
 3   skills            292167 non-null  object
 4   combined_text     292167 non-null  object
dtypes: object(5)
memory usage: 11.1+ MB


In [40]:
# Check missing values
missing_values = df.isnull().sum()
print("Missing Values:\n", missing_values)


Missing Values:
 Job Title           0
Job Description     0
Responsibilities    0
skills              0
combined_text       0
dtype: int64


In [41]:
# Fill missing skills with an empty string (instead of NaN)
df['skills'] = df['skills'].fillna("")

# Drop rows where Job Description is missing
df = df.dropna(subset=['Job Description'])

print("Missing Data Handled!")


Missing Data Handled!


# Dynamic Skill Extraction

In [49]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 292167 entries, 0 to 292166
Data columns (total 5 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   Job Title         292167 non-null  object
 1   Job Description   292167 non-null  object
 2   Responsibilities  292167 non-null  object
 3   skills            292167 non-null  object
 4   combined_text     292167 non-null  object
dtypes: object(5)
memory usage: 11.1+ MB


In [51]:
import pandas as pd
import re
from collections import Counter
from itertools import tee


In [52]:
def clean_skills(text):
    """Cleans and tokenizes skills while removing special characters."""
    if pd.isna(text):
        return []

    # Convert to lowercase and remove special characters
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9,\s-]+", " ", text)  # Keep commas & hyphens
    text = re.sub(r"\s+", " ", text).strip()  # Remove extra spaces

    # Split skills by commas or spaces
    skills_list = re.split(r",\s*|\s+", text)

    return skills_list


In [53]:
# Define stopwords that should NOT be counted as skills
stopwords = {"and", "with", "the", "in", "for", "on", "to", "of", "by", "from", "at", "or", "an", "a"}

def clean_skills(text):
    """Extracts skills while removing unwanted words."""
    if pd.isna(text):
        return []

    # Convert to lowercase and clean text
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9,]+", " ", text)  # Remove special characters

    # Split skills by commas or spaces
    skills_list = re.split(r",\s*|\s+", text)

    # Remove stopwords & very short words
    skills_list = [word.strip() for word in skills_list if len(word) > 2 and word not in stopwords]

    return skills_list


In [54]:
# Define words to remove only if they appear alone
generic_words = {"skills", "knowledge", "detail", "problem", "attention", "planning", "certifications"}

def refine_skills(skills_list):
    """Removes overly generic words while keeping multi-word skills."""
    refined = []
    for skill in skills_list:
        # Keep if it's a multi-word skill OR not in generic words
        if " " in skill or skill not in generic_words:
            refined.append(skill)
    return refined


In [55]:
def generate_ngrams(words, n=2):
    """Generate n-grams (bigrams, trigrams) from a list of words."""
    words = list(words)
    ngrams = zip(*[words[i:] for i in range(n)])
    return [" ".join(ngram) for ngram in ngrams]

def restore_multi_word_skills(skills_list):
    """Reconstructs multi-word skills from extracted single words."""
    if len(skills_list) < 2:
        return skills_list  # Nothing to combine if only one skill

    bigrams = generate_ngrams(skills_list, n=2)
    trigrams = generate_ngrams(skills_list, n=3)

    return list(set(skills_list + bigrams + trigrams))  # Combine single + bigrams + trigrams


In [63]:
# Apply skill extraction functions
df["Cleaned Skills"] = df["skills"].apply(clean_skills)
df["Refined Skills"] = df["Cleaned Skills"].apply(refine_skills)
df["Final Skills"] = df["Refined Skills"].apply(restore_multi_word_skills)

broad_words = {"development", "management", "design", "data", "tools", "practices", "user","solving", "communication" , "system", "proficiency","solving communication"}

def refine_final_skills(skills_list):
    """Removes broad words unless they are part of a phrase."""
    return [skill for skill in skills_list if " " in skill or skill not in broad_words]

df["Final Skills"] = df["Final Skills"].apply(refine_final_skills)


skill_replacements = {
    "html css": "html & css",
    "css javascript": "html, css & javascript",
    "html css javascript": "html, css & javascript"
}

def standardize_skills(skills_list):
    """Replaces variations of common multi-word skills with standardized names."""
    return [skill_replacements.get(skill, skill) for skill in skills_list]

df["Final Skills"] = df["Final Skills"].apply(standardize_skills)


# Flatten the list of all extracted skills
all_skills = [skill for sublist in df["Final Skills"] for skill in sublist]

# Count occurrences
skill_counts = Counter(all_skills)

# Get unique skill count
unique_skills_count = len(set(all_skills))

print(f"Total Extracted Skills: {len(all_skills)}")
print(f"Unique Skills Found: {unique_skills_count}")
print("Top 50 Most Frequent Skills:", skill_counts.most_common(50))


Total Extracted Skills: 11343259
Unique Skills Found: 1799
Top 50 Most Frequent Skills: [('security', 97839), ('troubleshooting', 73289), ('web', 55595), ('network', 52511), ('python', 52076), ('database', 51964), ('html, css & javascript', 48562), ('integration', 48556), ('testing', 41974), ('programming', 41867), ('java', 41860), ('analysis', 41822), ('languages', 41695), ('test', 38452), ('frameworks', 38445), ('server', 38313), ('optimization', 38149), ('sql', 38096), ('collaboration', 34761), ('systems', 34755), ('css', 34681), ('html', 34681), ('html & css', 34681), ('performance', 34657), ('recovery', 31526), ('network security', 31449), ('database management', 31335), ('frontend', 31322), ('automation', 31315), ('responsive', 31135), ('protocols', 28031), ('technologies', 28015), ('javascript', 27837), ('software', 27693), ('security protocols', 24549), ('cloud', 24530), ('monitoring', 24434), ('scripting', 24222), ('solving communication', 24213), ('api', 24164), ('modeling', 

In [62]:
df["Final Skills"] = df["Final Skills"].apply(lambda skills: ["AWS" if s == "aws azure" else s for s in skills])
df["Final Skills"] = df["Final Skills"].apply(lambda skills: ["Azure" if s == "aws azure" else s for s in skills])


In [61]:
df["Final Skills"] = df["Final Skills"].apply(lambda skills: ["backup & recovery" if s in {"backup", "backup recovery"} else s for s in skills])


In [65]:
import json

# Save skills as a JSON list
with open("extracted_tech_skills.json", "w") as file:
    json.dump(list(set(all_skills)), file, indent=4)

print("Skills list saved as JSON")


Skills list saved as JSON


In [67]:
import pandas as pd

# Convert unique skills into a DataFrame
skills_df = pd.DataFrame({"Skills": list(set(all_skills))})

# Save as CSV
skills_df.to_csv("extracted_tech_skills.csv", index=False)

print("Skills list saved as CSV")


Skills list saved as CSV!


In [66]:
# Save the final dataset
df.to_csv("cleaned_job_data.csv", index=False)

print("Dataset saved as CSV")


Dataset saved as CSV
